# Lab 09: Delta Lake (reference solution)

**Time:** about 2 h 30 min.

**Checklist questions this lab gives you evidence for:**<br><span style="color:#d93025"><b>7.1 Transaction log</b></span><br><span style="color:#d93025"><b>7.2 ACID in Delta Lake</b></span><br><span style="color:#d93025"><b>7.3 Schema enforcement</b></span><br><span style="color:#d93025"><b>7.4 OPTIMIZE</b></span><br><span style="color:#d93025"><b>7.5 Z-ordering</b></span><br><span style="color:#d93025"><b>7.6 UPDATE and DELETE on Delta</b></span><br><span style="color:#d93025"><b>7.7 MERGE INTO</b></span><br><span style="color:#d93025"><b>7.8 Snapshot isolation</b></span><br><span style="color:#d93025"><b>7.9 VACUUM</b></span><br><span style="color:#d93025"><b>7.10 Change Data Feed</b></span>

**Goal:** open the Delta transaction log, see what UPDATE, DELETE, MERGE, OPTIMIZE and VACUUM write to disk, and read the change feed. This lab uses Delta Lake 4.4 (the setup cell turns it on).

## Data information
A small online shop with **one table**. The setup cell below creates it in `data/orders.parquet` (1,000,000 orders from January to March 2024; seeded, so everyone gets the same rows):
* *order_id*; *customer_id* (empty for guest checkout), *customer_name*, *country*, *segment* (`retail`, `business` or `guest`)
* *product_id*, *category* (8 categories), *price* (price of one item)
* *quantity* (1 to 5) and *amount* = quantity × price, in dollars
* *status*: `paid`, `shipped`, `cancelled` or `returned`; *payment_method*: `card`, `cash` or `wallet`
* *order_ts* and *order_date*: when the order was placed; *tracking_code*: a 32-character code

Customer 1 is "Big Reseller Ltd": a business customer who orders in bulk (quantity 5) and places 30% of all orders. Some labs make a small lookup table from this one table (for example one row per customer); the notebook says when.

## How to work
* Write your answer in the code cell under each question, then run the check cell below it: ✅ correct, ❌ try again.
* When a question says *guess first*, write your guess as a comment before you run anything.
* Keep the Spark UI open (the setup cell prints its address).
* Stuck for more than 10 minutes? Read the documentation list at the top, then the reference solution.


## Documentation
Everything you need is on these pages. The questions do not repeat them: read, try things out in a cell, and find the answers yourself.

* [Delta transaction log (blog)](https://www.databricks.com/blog/2019/08/21/diving-into-delta-lake-unpacking-the-transaction-log.html)
* [Delta: table batch reads and writes](https://docs.delta.io/latest/delta-batch.html)
* [Delta: updates, deletes and merges](https://docs.delta.io/latest/delta-update.html)
* [Delta: schema validation and evolution](https://docs.delta.io/latest/delta-batch.html#schema-validation)
* [Delta: query an older snapshot](https://docs.delta.io/latest/delta-batch.html#query-an-older-snapshot-of-a-table-time-travel)
* [DeltaMergeBuilder](https://docs.delta.io/latest/api/python/spark/index.html)
* [Delta: optimizations](https://docs.delta.io/latest/optimizations-oss.html)
* [Delta: table utility commands](https://docs.delta.io/latest/delta-utility.html)
* [Delta: change data feed](https://docs.delta.io/latest/delta-change-data-feed.html)
* [SELECT: common table expressions](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-cte.html)
* [Window functions (SQL)](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-window.html)

## Setup
Run the cell below once (its code is hidden: click it to expand). It creates the data and starts Spark.

In [ ]:
# Setup: run this cell once. It creates the shop data in data/ and starts Spark.
# Nothing to solve here, and you do not need to read it.
import contextlib, hashlib, io, json, os, re, shutil, sys, time, urllib.request
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this kernel's Python
from pyspark.sql import SparkSession, Window, functions as F

# ---------------------------------------------------------------- the shop data ---------
# A small online shop with ONE table, generated from seeded hash functions: every machine gets the same rows.
COUNTRIES = ["Uzbekistan", "Germany", "USA", "India", "Brazil", "Japan"]
CATEGORIES = ["books", "toys", "garden", "sports", "kitchen", "beauty", "music", "office"]
N_CUSTOMERS, N_PRODUCTS, N_ORDERS = 10_000, 200, 1_000_000

def _u(col, salt):
    """Pseudo-random number in [0, 1) computed from a column (same result on every machine)."""
    return F.pmod(F.xxhash64(col, F.lit(salt)), F.lit(1_000_000)) / 1_000_000.0

def _pick(u, options):
    return F.element_at(F.array(*[F.lit(o) for o in options]), (F.floor(u * len(options)) + 1).cast("int"))

def make_orders(spark, n=N_ORDERS, dirty=False):
    i = F.col("id")
    cust = (F.when(_u(i, 11) < 0.02, F.lit(None).cast("long"))                        # guest checkout: no customer
             .when(_u(i, 10) < 0.30, F.lit(1).cast("long"))                           # the big reseller: 30% of orders
             .otherwise((F.floor(_u(i, 12) * (N_CUSTOMERS - 1)) + 2).cast("long")))
    name = F.when(cust == 1, "Big Reseller Ltd").when(cust.isNotNull(), F.concat(F.lit("Customer "), cust.cast("string")))
    country = _pick(F.when(cust.isNotNull(), _u(cust, 1)).otherwise(_u(i, 31)), COUNTRIES)
    segment = F.when(cust.isNull(), "guest").when((cust == 1) | (_u(cust, 3) < 0.2), "business").otherwise("retail")
    product = (F.floor(_u(i, 14) * N_PRODUCTS) + 1).cast("int")
    category = F.element_at(F.array(*[F.lit(c) for c in CATEGORIES]), (product % 8 + 1).cast("int"))
    price = F.round((product * 37) % 195 + 5.99, 2)
    quantity = F.when(cust == 1, F.lit(5)).otherwise(F.floor(_u(i, 13) * 5) + 1).cast("int")   # the reseller always orders in bulk
    ts = F.lit("2024-01-01 00:00:00").cast("timestamp") + F.make_interval(secs=F.floor(_u(i, 15) * 91 * 86400).cast("int"))
    u = _u(i, 17)
    status = F.when(u < 0.55, "paid").when(u < 0.80, "shipped").when(u < 0.92, "cancelled").otherwise("returned")
    v = _u(i, 18)
    payment = F.when(v < 0.60, "card").when(v < 0.85, "cash").otherwise("wallet")
    df = spark.range(0, n, 1, 4).select(
        (i + 1).alias("order_id"), cust.alias("customer_id"), name.alias("customer_name"), country.alias("country"), segment.alias("segment"),
        product.alias("product_id"), category.alias("category"), price.alias("price"), quantity.alias("quantity"),
        F.round(quantity * price, 2).alias("amount"), status.alias("status"), payment.alias("payment_method"),
        ts.alias("order_ts"), F.md5(i.cast("string")).alias("tracking_code"))
    df = df.withColumn("order_date", F.to_date("order_ts"))
    if dirty:    # the monthly files of the final project contain mistakes on purpose
        df = df.withColumn("amount", F.when(_u(F.col("order_id"), 19) < 0.008, -F.col("amount")).otherwise(F.col("amount")))
        df = df.withColumn("quantity", F.when(_u(F.col("order_id"), 20) < 0.006, 0).otherwise(F.col("quantity")))
        df = df.unionByName(df.filter(_u(F.col("order_id"), 21) < 0.005))              # exact duplicates
    return df.select("order_id", "customer_id", "customer_name", "country", "segment", "product_id", "category", "price", "quantity",
                     "amount", "status", "payment_method", "order_ts", "order_date", "tracking_code")

def _create_data(spark):
    os.makedirs("data/monthly", exist_ok=True)
    make_orders(spark).repartition(4).write.mode("overwrite").parquet("data/orders.parquet")
    dirty = make_orders(spark, dirty=True)
    wrong = _u(F.col("order_id"), 22) < 0.003          # a few orders land in the wrong month's file
    month = F.month("order_date")
    file_month = F.when(wrong, month % 3 + 1).otherwise(month)
    for m in (1, 2, 3):
        (dirty.filter(file_month == m).coalesce(1).write.mode("overwrite")
              .parquet(f"data/monthly/orders_2024-{m:02d}.parquet"))

# ---------------------------------------------------------------- the Spark session ----
def new_session(master="local[4]", driver_memory="2g", delta=False, avro=False, aqe=False, conf=None):
    builder = (SparkSession.builder.master(master).appName("day09_lab")
               .config("spark.driver.memory", driver_memory)
               .config("spark.ui.showConsoleProgress", "false")
               .config("spark.sql.adaptive.enabled", str(aqe).lower())   # off = exact job and stage counts
               .config("spark.sql.session.timeZone", "UTC")                # same dates and hours for everyone
               .config("spark.sql.warehouse.dir", os.path.abspath("spark-warehouse"))
               # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this.
               .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false"))
    for k, v in (conf or {}).items():
        builder = builder.config(k, v)
    if delta:
        from delta import configure_spark_with_delta_pip
        builder = (builder.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
                          .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        builder = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.spark:spark-avro_2.13:4.2.0"] if avro else None)
    elif avro:
        builder = builder.config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.2.0")
    s = builder.getOrCreate()
    s.sparkContext.setLogLevel("ERROR")
    return s

for _d in ("spark-warehouse", "metastore_db"):
    shutil.rmtree(_d, ignore_errors=True)               # a clean catalog on every start
spark = new_session(delta=True)
def _data_ok():
    """True if the single-table shop data is already there (an older three-table version is rebuilt)."""
    try:
        return "customer_name" in spark.read.parquet("data/orders.parquet").columns and os.path.exists("data/monthly/orders_2024-03.parquet")
    except Exception:
        return False

if not _data_ok():
    print("creating the shop data (about 30 seconds, once)...")
    _create_data(spark)
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

try:
    from delta.tables import DeltaTable   # for the Delta labs
except ImportError:
    pass

# ---------------------------------------------------------------- helpers ----------------
def _tracker():
    return spark.sparkContext.statusTracker()

def _run_group(action):
    sc = spark.sparkContext
    group = f"g{time.time_ns()}"
    sc.setJobGroup(group, group)
    try:
        action()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    return [_tracker().getJobInfo(j) for j in sorted(_tracker().getJobIdsForGroup(group))]

def jobs_run(action):
    """How many Spark jobs calling action() starts."""
    return len(_run_group(action))

def stages_run(action):
    """How many stages those jobs had in total."""
    return sum(len(j.stageIds) for j in _run_group(action))

def last_stage_tasks(action):
    """Number of tasks in the last stage action() ran."""
    jobs = _run_group(action)
    return _tracker().getStageInfo(max(s for j in jobs for s in j.stageIds)).numTasks

def _api(path):
    sc = spark.sparkContext
    return json.load(urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}{path}"))

api = _api   # read anything from the Spark UI's REST API, e.g. api("/executors")

def _stage_rows(action):
    jobs = _run_group(action)
    ids = {s for j in jobs for s in j.stageIds}
    return [s for s in _api("/stages?length=2000") if s["stageId"] in ids]

def shuffle_bytes(action):
    """Total shuffle bytes written by the stages action() ran."""
    return sum(s["shuffleWriteBytes"] for s in _stage_rows(action))

def spill_bytes(action):
    """(memory, disk) bytes spilled by the stages action() ran."""
    rows = _stage_rows(action)
    return sum(s["memoryBytesSpilled"] for s in rows), sum(s["diskBytesSpilled"] for s in rows)

def skew_ratio(action):
    """Slowest task / median task, in the stage with the most tasks that action() ran."""
    rows = _stage_rows(action)
    st = max(rows, key=lambda s: s["numTasks"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=5000")
    times = sorted(t["taskMetrics"]["executorRunTime"] for t in tasks if t.get("taskMetrics"))
    return round(times[-1] / max(times[len(times) // 2], 1), 1)

def failed_attempts():
    """Failed task attempts in the most recent failed stage (from the Spark UI's REST API)."""
    failed = _api("/stages?status=failed")
    if not failed:
        return 0
    st = max(failed, key=lambda s: s["stageId"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=500")
    return sum(1 for t in tasks if t["status"] == "FAILED")

def sql_metric(action, metric):
    """Value of one scan metric (for example 'number of files read') of the query action() runs."""
    desc = f"q{time.time_ns()}"
    spark.sparkContext.setJobDescription(desc)
    try:
        action()
    finally:
        spark.sparkContext.setJobDescription(None)
    for ex in _api("/sql?details=true&planDescription=false&length=2000"):
        if ex.get("description") == desc:
            vals = [m["value"] for n in ex["nodes"] for m in n["metrics"] if m["name"] == metric]
            return vals[0] if vals else None

def noop(df):
    """Run the whole query but throw the result away."""
    df.write.format("noop").mode("overwrite").save()

def plan(df, mode="formatted"):
    """explain() output as a string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

def count_nodes(df, *names):
    """How often each operator name appears in the simple plan."""
    text = plan(df, "simple")
    return {n: len(re.findall(r"\b" + re.escape(n) + r"\b", text)) for n in names}

def optimized(df):
    """The optimized logical plan as a string."""
    return df._jdf.queryExecution().optimizedPlan().toString()

def operators(df):
    """Operator names of the physical plan tree, top to bottom."""
    tree = plan(df).split("\n\n")[0].splitlines()[1:]
    return [re.sub(r"\s*\(\d+\)$", "", line).strip(" +-*:") for line in tree]

def rules_fired(df):
    """Names of the Catalyst rules that changed this query's plan."""
    qe = df._jdf.queryExecution(); qe.executedPlan()
    it, fired = qe.tracker().rules().iterator(), set()
    while it.hasNext():
        kv = it.next()
        if kv._2().numEffectiveInvocations() > 0:
            fired.add(kv._1().split(".")[-1].split("$")[-1])
    return fired

def estimate(df):
    """(sizeInBytes, rowCount or None) the optimizer estimates for df."""
    stats = df._jdf.queryExecution().optimizedPlan().stats()
    return int(str(stats.sizeInBytes())), (int(str(stats.rowCount().get())) if stats.rowCount().isDefined() else None)

def partition_sizes(df):
    """Rows per partition, largest first."""
    return sorted((r["count"] for r in df.groupBy(F.spark_partition_id().alias("p")).count().collect()), reverse=True)

def timeit(action):
    start = time.perf_counter(); action(); return round(time.perf_counter() - start, 2)

def folder_mb(path):
    files = [os.path.join(d, f) for d, _, fs in os.walk(path) for f in fs] if os.path.isdir(path) else [path]
    return round(sum(os.path.getsize(f) for f in files) / 1e6, 1)

# ---------------------------------------------------------------- checks -----------------
_EXPECTED = {
 "9.1": "7f424065ae2cf5e2",
 "9.2": "32c0a48ee01e9b13",
 "9.3": "89ec914809c3cf22",
 "9.4": "26487d3edd2ef303",
 "9.5": "c934222fc335b35c",
 "9.8": "08f157ae0e3ed779",
 "9.9": "379c454e341a4175"
}
_RECORD = os.environ.get("LAB_RECORD")

def _norm(v):
    if isinstance(v, bool) or v is None: return v
    if isinstance(v, int): return v
    if isinstance(v, float): return round(v, 2)
    if isinstance(v, str): return v.strip()
    if isinstance(v, dict): return {str(k): _norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
    if isinstance(v, (list, tuple)): return [_norm(x) for x in v]
    if isinstance(v, (set, frozenset)): return sorted((_norm(x) for x in v), key=json.dumps)
    if hasattr(v, "item"): return _norm(v.item())
    if type(v).__name__ == "Decimal": return round(float(v), 2)
    raise TypeError(f"check() needs plain Python values, got {type(v).__name__}")

def check(task, value):
    normal = _norm(value)
    if _RECORD:
        rec = json.load(open(_RECORD)) if os.path.exists(_RECORD) else {}
        rec[task] = normal; json.dump(rec, open(_RECORD, "w"))
    ok = hashlib.sha256(json.dumps(normal).encode()).hexdigest()[:16] == _EXPECTED.get(task)
    print(f"✅ {task} correct" if ok else f"❌ {task} not yet: check your code and the question again")

def check_true(task, condition, hint=""):
    print(f"✅ {task} correct" if condition else f"❌ {task} not yet. {hint}")


Load the data the way Lab 01 taught you (given):

In [ ]:
orders = spark.read.parquet("data/orders.parquet")

## A. The table and its log

**9.1 Create a Delta table and open its log.** Write the orders with `order_id <= 100000` as a Delta table to `delta/orders` (path-based, `format("delta")`). Open the first commit file `delta/orders/_delta_log/00000000000000000000.json`: one JSON action per line.
* *actions* = the sorted list of action names that appear (`add`, `metaData`, ...)
* *total_records* = the sum of `numRecords` from the `stats` of all `add` actions

In [ ]:
shutil.rmtree("delta", ignore_errors=True)
small = orders.filter(F.col("order_id") <= 100000)
small.write.format("delta").save("delta/orders")
log0 = [json.loads(line) for line in open("delta/orders/_delta_log/00000000000000000000.json")]
actions = sorted({k for a in log0 for k in a})
total_records = sum(json.loads(a["add"]["stats"])["numRecords"] for a in log0 if "add" in a)
actions, total_records

In [ ]:
check("9.1", [actions, total_records])

**9.2 What do UPDATE and DELETE write?** With `DeltaTable.forPath(spark, "delta/orders")`: set *status* to `void` for all `cancelled` orders (`update`), then delete all `returned` orders (`delete`). Read the operation metrics from `table.history()` (`operationMetrics`): *updated* = `numUpdatedRows`, *deleted* = `numDeletedRows` (ints). Open the newest commit file: *last_actions* = sorted list of its action names. Notice that the old Parquet files are still in the folder.

In [ ]:
table = DeltaTable.forPath(spark, "delta/orders")
table.update("status = 'cancelled'", {"status": "'void'"})
table.delete("status = 'returned'")
metrics = {r["operation"]: r["operationMetrics"] for r in table.history().select("operation", "operationMetrics").collect()}
updated, deleted = int(metrics["UPDATE"]["numUpdatedRows"]), int(metrics["DELETE"]["numDeletedRows"])
last = sorted(os.listdir("delta/orders/_delta_log"))
last_file = [f for f in last if f.endswith(".json")][-1]
last_actions = sorted({k for line in open(f"delta/orders/_delta_log/{last_file}") for k in json.loads(line)})
updated, deleted, last_actions

In [ ]:
check("9.2", [updated, deleted, last_actions])

**9.3 Schema enforcement.** Append 10 orders with an extra column *extra* (`small.limit(10).withColumn("extra", F.lit(1))`) to the table. The write must fail: *error_ok* = True if the error message contains `DELTA_METADATA_MISMATCH`. Then append again with `option("mergeSchema", "true")` and put the table's columns in *cols_after*.

In [ ]:
extra = small.limit(10).withColumn("extra", F.lit(1))
try:
    extra.write.format("delta").mode("append").save("delta/orders")
    error_ok = False
except Exception as e:
    error_ok = "DELTA_METADATA_MISMATCH" in str(e)
extra.write.format("delta").mode("append").option("mergeSchema", "true").save("delta/orders")
cols_after = spark.read.format("delta").load("delta/orders").columns
error_ok, cols_after

In [ ]:
check("9.3", [error_ok, cols_after])

**9.4 Time travel and snapshots.** Read version 0 of the table (`option("versionAsOf", 0)`) and the current version. Put the row counts in *counts* as `(v0, now)`. Why does a long-running query never see a half-finished commit?

In [ ]:
v0 = spark.read.format("delta").option("versionAsOf", 0).load("delta/orders")
now = spark.read.format("delta").load("delta/orders")
counts = (v0.count(), now.count())
counts

In [ ]:
check("9.4", counts)

## B. MERGE

**9.5 An idempotent upsert.** Create the table `delta/orders_merge` from the orders with `order_id <= 1000`. *source* has 5 existing orders with a new *amount* (`order_id <= 5`, amount + 1) and 5 new orders (`order_id + 10000000`). Upsert it with `merge` on *order_id*: update only when the *amount* changed, insert when new. Read `numTargetRowsUpdated` and `numTargetRowsInserted` from `history(1)` after the first MERGE (*first*) and after running the same MERGE again (*second*), as ints `(updated, inserted)`.

Then add a second source row for one key and read the error: *dup_error* = True if it contains `DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE`.

In [ ]:
shutil.rmtree("delta/orders_merge", ignore_errors=True)
orders.filter("order_id <= 1000").write.format("delta").save("delta/orders_merge")
mtable = DeltaTable.forPath(spark, "delta/orders_merge")
changed = orders.filter("order_id <= 5").withColumn("amount", F.col("amount") + 1)
new = orders.filter("order_id <= 5").withColumn("order_id", F.col("order_id") + 10000000)
source = changed.unionByName(new)

def upsert(src):
    (mtable.alias("t").merge(src.alias("s"), "t.order_id = s.order_id")
        .whenMatchedUpdateAll(condition="t.amount <> s.amount")
        .whenNotMatchedInsertAll().execute())
    m = mtable.history(1).select("operationMetrics").first()[0]
    return (int(m["numTargetRowsUpdated"]), int(m["numTargetRowsInserted"]))

first = upsert(source)
second = upsert(source)
try:
    upsert(source.unionByName(source.limit(1).withColumn("amount", F.lit(1.0))))
    dup_error = False
except Exception as e:
    dup_error = "DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE" in str(e)
first, second, dup_error

In [ ]:
check("9.5", [first, second, dup_error])

## C. Maintenance: OPTIMIZE, Z-order, VACUUM

**9.6 Compact and Z-order.** Build two tables `delta/a` and `delta/b` the same way: append 50 batches of 20,000 orders (`order_id` from `i * 20000 + 1` to `(i + 1) * 20000`), which makes about 200 small files each. Set `spark.databricks.delta.optimize.maxFileSize` to `str(8 * 1024 * 1024)`. Compact *a* with `optimize().executeCompaction()` and Z-order *b* by *customer_id* with `optimize().executeZOrderBy("customer_id")`. Then count the files a filter `customer_id = 42` has to read on each (`len(df.filter(...).inputFiles())`): *files_a* and *files_b*, and the total number of files of each: *total_a*, *total_b*.

In [ ]:
spark.conf.set("spark.databricks.delta.optimize.maxFileSize", str(8 * 1024 * 1024))
for name in ("a", "b"):
    shutil.rmtree(f"delta/{name}", ignore_errors=True)
    for i in range(50):
        (orders.filter((F.col("order_id") > i * 20000) & (F.col("order_id") <= (i + 1) * 20000))
               .write.format("delta").mode("append").save(f"delta/{name}"))
DeltaTable.forPath(spark, "delta/a").optimize().executeCompaction()
DeltaTable.forPath(spark, "delta/b").optimize().executeZOrderBy("customer_id")
da, db = spark.read.format("delta").load("delta/a"), spark.read.format("delta").load("delta/b")
files_a, files_b = len(da.filter("customer_id = 42").inputFiles()), len(db.filter("customer_id = 42").inputFiles())
total_a, total_b = len(da.inputFiles()), len(db.inputFiles())
files_a, total_a, files_b, total_b

In [ ]:
check_true("9.6", files_a == total_a and files_b <= 2 and total_b < 20, "compaction alone reads every file; Z-ordering lets the filter read only one or two")

**9.7 VACUUM breaks old versions.** Turn the safety check off for this practice table (`spark.databricks.delta.retentionDurationCheck.enabled` = false) and run `table.vacuum(0)` on `delta/orders`. Then read version 0 and compute `sum(amount)` (not `count()`: Delta answers a count from its log). *vacuum_error* = True if reading version 0 now fails with an error that mentions `FILE_NOT_EXIST` or `No such file`.

In [ ]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
DeltaTable.forPath(spark, "delta/orders").vacuum(0)
try:
    spark.read.format("delta").option("versionAsOf", 0).load("delta/orders").agg(F.sum("amount")).collect()
    vacuum_error = False
except Exception as e:
    vacuum_error = "FILE_NOT_EXIST" in str(e) or "No such file" in str(e)
vacuum_error

In [ ]:
check_true("9.7", vacuum_error, "after VACUUM with 0 hours the files of version 0 are gone")

## D. Change Data Feed

**9.8 Read the change feed of one MERGE.** Create the table `delta/cdf` from the orders with `order_id <= 100` and turn on the change feed with an `ALTER TABLE ... SET TBLPROPERTIES (delta.enableChangeDataFeed = true)` statement on the table's absolute path (a path table is written as delta.`/the/path`). *src* (given) has: orders 1 to 3 with a new amount, orders 4 and 5 marked `delete_flag`, and two new orders. Run one MERGE: delete the rows with `delete_flag`, update the amount of the other matched rows, insert the new ones. Read the change feed from the version after enabling it (`option("readChangeFeed", "true")`, `startingVersion`) and count the rows per `_change_type` into a dict *changes*.

In [ ]:
src = (orders.filter("order_id <= 5").withColumn("amount", F.col("amount") + 1).withColumn("delete_flag", F.col("order_id") >= 4)
       .unionByName(orders.filter("order_id in (6, 7)").withColumn("order_id", F.col("order_id") + 20000000)
                    .withColumn("delete_flag", F.lit(False))))

shutil.rmtree("delta/cdf", ignore_errors=True)
orders.filter("order_id <= 100").write.format("delta").save("delta/cdf")
spark.sql(f"ALTER TABLE delta.`{os.path.abspath('delta/cdf')}` SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
ctable = DeltaTable.forPath(spark, "delta/cdf")
start = ctable.history(1).first()["version"] + 1
(ctable.alias("t").merge(src.alias("s"), "t.order_id = s.order_id")
    .whenMatchedDelete(condition="s.delete_flag = true")
    .whenMatchedUpdate(set={"amount": "s.amount"})
    .whenNotMatchedInsertAll().execute())
feed = spark.read.format("delta").option("readChangeFeed", "true").option("startingVersion", start).load("delta/cdf")
changes = {r["_change_type"]: r["count"] for r in feed.groupBy("_change_type").count().collect()}
changes

In [ ]:
check("9.8", changes)

## E. Coding practice in Spark SQL

The same Spark you used with DataFrames also speaks SQL. Solve this one with `spark.sql` on a view, using a CTE and a window function.

**9.9 The top 2 customers per country (SQL).** Register *orders* as a temp view. In one `spark.sql` query, use a CTE that sums the *amount* of *paid* orders per customer and country (no guests), then rank the customers inside each country by that revenue (largest first, ties by *customer_id*) with `row_number()` and keep ranks 1 and 2. *top_per_country* = list of `(country, customer_id, revenue)` sorted by *country*, then rank, revenue rounded to 2 decimals.

In [ ]:
orders.createOrReplaceTempView("orders")
sql_top = spark.sql("""
    WITH revenue AS (
        SELECT country, customer_id, SUM(amount) AS revenue
        FROM orders
        WHERE status = 'paid' AND customer_id IS NOT NULL
        GROUP BY country, customer_id
    ), ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY country ORDER BY revenue DESC, customer_id) AS rn FROM revenue
    )
    SELECT country, customer_id, ROUND(revenue, 2) AS revenue FROM ranked WHERE rn <= 2 ORDER BY country, rn
""")
top_per_country = [(r["country"], r["customer_id"], r["revenue"]) for r in sql_top.collect()]
top_per_country

In [ ]:
check("9.9", top_per_country)

---
Done? In the Theory part of Day 9, answer the red checklist topics aloud, with the log files and metrics from this lab as evidence.